# Sweeps for Design Analysis

**Audience:** Design and data-analysis users who want repeatable parameter studies and Monte Carlo inputs from Python.

This notebook builds deterministic sweep points, inspects generated netlists, and shows how the same output specs are reused across runs.

In [ ]:
import pandas as pd

from xyce_py import (
    MonteCarloParameter,
    NormalDistribution,
    OutputSpec,
    SweepParameter,
    UniformDistribution,
    XyceMonteCarloSweep,
    XyceParameterSweep,
)

## Base netlist template

The sweep runner inserts explicit `.PARAM` lines after the title line. The base netlist should not already define the same `.PARAM` names.

In [ ]:
base_netlist = """* divider sweep
V1 1 0 DC 10
R1 1 2 {RLOAD}
R2 2 0 {RBOTTOM}
.OP
.PRINT DC FORMAT=CSV FILE=out.csv V(2)
.END
"""

waveform_outputs = (OutputSpec.csv("waveforms", "out.csv"),)

## Cartesian parameter sweep

`XyceParameterSweep.points()` returns all combinations before any simulator run is launched.

In [ ]:
sweep = XyceParameterSweep(
    "divider-grid",
    base_netlist,
    parameters=(
        SweepParameter("RLOAD", ["500", "1k", "2k"]),
        SweepParameter("RBOTTOM", ["1k", "2k"]),
    ),
    output_specs=waveform_outputs,
)

grid_points = sweep.points()
grid_point_frame = pd.DataFrame([point.parameters for point in grid_points])
grid_point_frame.insert(0, "index", [point.index for point in grid_points])
print(grid_point_frame)
print(sweep.netlist_for_point(grid_points[0]))

## Deterministic Monte Carlo points

Monte Carlo sweeps use explicit sampled `.PARAM` values with a fixed seed, so generated points are reproducible.

In [ ]:
monte_carlo = XyceMonteCarloSweep(
    "divider-monte-carlo",
    base_netlist,
    parameters=(
        MonteCarloParameter("RLOAD", UniformDistribution(900.0, 1100.0)),
        MonteCarloParameter("RBOTTOM", NormalDistribution(1000.0, 25.0)),
    ),
    samples=5,
    seed=7,
    output_specs=waveform_outputs,
)

monte_carlo_points = monte_carlo.points()
monte_carlo_frame = pd.DataFrame([point.parameters for point in monte_carlo_points])
monte_carlo_frame.insert(0, "index", [point.index for point in monte_carlo_points])
print(monte_carlo_frame)
print(monte_carlo.netlist_for_point(monte_carlo_points[0]))